# 🎨 TechnoCrazy — Generador de Imágenes de Rafael (Flux LoRA)
**Entrena un modelo con tus fotos → genera imágenes tuyas en cualquier escena**

### Dos modos:
- **Modo A:** Entrenar LoRA (solo primera vez, ~30 min, ~$2 en Replicate)
- **Modo B:** Generar imágenes con el LoRA ya entrenado ($0 en Colab)

> Runtime → T4 GPU obligatorio

In [ ]:
# CELDA 1 — Instalar dependencias
!pip install -q diffusers transformers accelerate safetensors
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU"}')

In [ ]:
# CELDA 2 — SUBIR TUS FOTOS DE REFERENCIA (15-25 fotos)
from google.colab import files
import os, shutil

os.makedirs('/content/fotos_rafael', exist_ok=True)

print('📸 Sube tus fotos de TechnoCrazy (15-25 fotos):')
print('   - Variedad de expresiones y ángulos')
print('   - JPG o PNG, mínimo 512x512px')
print('   - Las fotos profesionales de TechnoCrazy son perfectas')

uploaded = files.upload()
for nombre, contenido in uploaded.items():
    shutil.copy(nombre, f'/content/fotos_rafael/{nombre}')

fotos = os.listdir('/content/fotos_rafael')
print(f'\n✅ {len(fotos)} fotos cargadas')

In [ ]:
# CELDA 3A — ENTRENAR LORA EN REPLICATE (~$2, solo primera vez)
# Si ya tienes el LoRA entrenado, salta a la CELDA 4

!pip install -q replicate
import replicate, zipfile, os

# ⬇️ PON TU API KEY DE REPLICATE
REPLICATE_API_TOKEN = "r8_XXXXXXXXXX"  # replicate.com → Account → API Tokens
os.environ['REPLICATE_API_TOKEN'] = REPLICATE_API_TOKEN

# Comprimir fotos en ZIP
zip_path = '/content/fotos_rafael.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for foto in os.listdir('/content/fotos_rafael'):
        zf.write(f'/content/fotos_rafael/{foto}', foto)
print(f'ZIP creado: {zip_path}')

# Entrenar LoRA
print('🚀 Enviando a Replicate para entrenar... (~30 min, ~$2)')
training = replicate.trainings.create(
    version="ostris/flux-dev-lora-trainer:e5a5bc82",
    input={
        "input_images": open(zip_path, 'rb'),
        "trigger_word": "RAFAELNAVARRO",
        "steps": 1000,
        "learning_rate": 0.0004,
        "batch_size": 1,
        "lora_rank": 16,
        "caption_dropout_rate": 0.05,
    },
    destination="rafaelnavarro/rafael-lora"  # Cambia 'rafaelnavarro' por tu usuario
)

print(f'Training ID: {training.id}')
print(f'Estado: {training.status}')
print('\nGuarda el Training ID — lo necesitas cuando termine')
print('Recibirás email cuando esté listo (~30 min)')

In [ ]:
# CELDA 4 — GENERAR IMÁGENES con el LoRA (gratis en Colab)
from diffusers import FluxPipeline
import torch
from PIL import Image
from IPython.display import display

# ⬇️ ESCENAS PARA GENERAR (modifica según lo que necesites)
PROMPTS = [
    "RAFAELNAVARRO sitting at a modern desk with multiple screens showing code, dark studio with blue LED lights, TechnoCrazy logo in background, professional photography",
    "RAFAELNAVARRO standing in front of a whiteboard with AI flowcharts, confident pose, arms crossed, black leather jacket, bright office",
    "RAFAELNAVARRO holding a glowing USB drive toward camera, dark background with purple lighting, TechnoCrazy branding visible",
    "RAFAELNAVARRO at laptop in modern cafe, thinking expression, finger on chin, casual professional look",
    "RAFAELNAVARRO celebrating success, fist pump, screens with analytics in background, energy and excitement",
    "RAFAELNAVARRO in airport lounge with laptop, executive class, global entrepreneur lifestyle",
    "RAFAELNAVARRO pointing at camera, direct gaze, black leather jacket, white background, motivational pose",
]

# Cargar pipeline Flux
print('Cargando modelo Flux... (primera vez tarda ~5 min)')
pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-dev',
    torch_dtype=torch.float16
)
pipe = pipe.to('cuda')

# Cargar el LoRA de Rafael (después de entrenarlo en Replicate)
# pipe.load_lora_weights('rafaelnavarro/rafael-lora')  # Descomentar cuando tengas el LoRA

print('\n🎨 Generando imágenes...')
os.makedirs('/content/imagenes_rafael', exist_ok=True)

imagenes = []
for i, prompt in enumerate(PROMPTS):
    print(f'Generando {i+1}/{len(PROMPTS)}...')
    image = pipe(
        prompt=prompt,
        height=1792, width=1024,  # Vertical 9:16 para Reels
        num_inference_steps=20,
        guidance_scale=3.5
    ).images[0]
    
    path = f'/content/imagenes_rafael/rafael_{i+1:02d}.jpg'
    image.save(path, quality=95)
    imagenes.append(path)
    display(image.resize((270, 480)))

print(f'\n✅ {len(imagenes)} imágenes generadas')

In [ ]:
# CELDA 5 — GUARDAR EN GOOGLE DRIVE
from google.colab import drive
import shutil, os
from datetime import datetime

drive.mount('/content/drive')
fecha = datetime.now().strftime('%Y-%m-%d')
carpeta = f'/content/drive/MyDrive/TechnoCrazy/Contenido/Imagenes/{fecha}'
os.makedirs(carpeta, exist_ok=True)

for img in imagenes:
    shutil.copy(img, carpeta)

print(f'✅ {len(imagenes)} imágenes guardadas en Drive: {carpeta}')
print('n8n las tomará automáticamente para publicar')